# Análisis Exploratorio de Datos — RSNA Pneumonia Detection Challenge

**Proyecto Integrador en IA (MIAR0545) · Semana 2**
[retirado] [retirado] [retirado] [retirado] e [retirado] [retirado] [retirado] [retirado] · UEES

Proyecto: *Copiloto de triaje para neumonía en adultos*

## Sobre qué se hace el análisis

El dataset son imágenes, no una tabla. Lo que se explora aquí es la **tabla de
metadatos** que el propio pipeline del proyecto extrae de cada radiografía:
edad, sexo, posición de la placa, dimensiones, peso del archivo y la etiqueta.

Esa tabla la genera `scripts/extraer_metadatos.py`, que lee **solo las
cabeceras** de los 26 684 DICOM (`stop_before_pixels=True`). Por eso recorrer
el dataset completo tarda 26 segundos en vez de la media hora que costaría
decodificar cada imagen.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.width", 120)

RUTA = "metadatos_eda.csv"   # generado por scripts/extraer_metadatos.py
d = pd.read_csv(RUTA)
d["diagnostico"] = d.etiqueta.map({0: "Sin opacidad", 1: "Con opacidad"})
orden = ["Normal", "No Lung Opacity / Not Normal", "Lung Opacity"]

print(f"{len(d):,} pacientes  |  {d.shape[1]} columnas")
d.head()

## 1. Estadísticas descriptivas y valores faltantes

In [ ]:
print("=== Variables numéricas ===")
print(d[["edad_num", "filas", "columnas", "bytes", "n_cajas"]].describe().round(1))

print()
print("=== Variables categóricas ===")
for col in ["sexo", "vista", "clase", "modalidad", "interpretacion"]:
    print(f"{col:16s}", d[col].value_counts().to_dict())

print()
print("=== Valores faltantes ===")
faltantes = d.isna().sum()
print(faltantes[faltantes > 0].to_dict() or "Ninguno: el dataset está completo.")

## 2. La variable objetivo está desbalanceada

Un paciente es positivo si su radiografía tiene al menos una caja de opacidad.
La proporción real es **22,5%**, no el 50% que se suele suponer al leer la
descripción del reto.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=d, y="clase", order=orden, ax=ejes[0], hue="clase", legend=False)
ejes[0].set(title="Clase detallada (3 niveles)", xlabel="pacientes", ylabel="")
ejes[0].set_xlim(0, 15000)
for c in ejes[0].containers:
    ejes[0].bar_label(c, fmt=lambda v: f"{v:,.0f}  ({v/len(d):.1%})", padding=3, fontsize=9)

sns.countplot(data=d, x="diagnostico", ax=ejes[1], hue="diagnostico", legend=False)
ejes[1].set(title="Etiqueta binaria del modelo", xlabel="", ylabel="pacientes")
for c in ejes[1].containers:
    ejes[1].bar_label(c, fmt=lambda v: f"{v:,.0f} ({v/len(d):.1%})", padding=3, fontsize=9)

fig.suptitle("Desbalance de clases: 22,5% de positivos", y=1.02, weight="bold")
plt.tight_layout()
plt.show()

## 3. Edad: valores atípicos que contradicen el alcance declarado

El alcance del proyecto dice "adultos". La edad registrada en el DICOM dice
otra cosa en parte del padrón, y además contiene valores imposibles.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=d, x="edad_num", hue="diagnostico", bins=60, ax=ejes[0],
             element="step", common_norm=False, stat="density")
ejes[0].axvline(18, color="crimson", ls="--", lw=1.5)
ejes[0].text(18.5, ejes[0].get_ylim()[1] * 0.9, "18 años", color="crimson", fontsize=9)
ejes[0].set(title="Distribución de la edad", xlabel="edad (años)", ylabel="densidad")

sns.boxplot(data=d, x="edad_num", y="clase", order=orden, ax=ejes[1],
            hue="clase", legend=False)
ejes[1].set(title="Edad por clase (los puntos son atípicos)",
            xlabel="edad (años)", ylabel="")
plt.tight_layout()
plt.show()

menores = d[d.edad_num < 18]
mayores = d[d.edad_num > 100]
print(f"Menores de 18 años: {len(menores):,} ({len(menores)/len(d):.2%})")
print(f"Mayores de 100:     {len(mayores):,} — edades {sorted(mayores.edad_num.unique())}")

## 4. Hallazgo principal: la posición de la placa predice el diagnóstico

Este es el resultado con más consecuencias para el modelo.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

tasa = d.groupby("vista").etiqueta.mean().mul(100).reset_index()
sns.barplot(data=tasa, x="vista", y="etiqueta", ax=ejes[0], hue="vista", legend=False)
ejes[0].axhline(d.etiqueta.mean() * 100, color="black", ls=":", lw=1.5)
ejes[0].text(1.25, d.etiqueta.mean() * 100 + 1, "media global 22,5%", fontsize=9)
ejes[0].set(title="Tasa de opacidad según la posición", xlabel="", ylabel="% con opacidad")
for c in ejes[0].containers:
    ejes[0].bar_label(c, fmt="%.1f%%", padding=3)

sns.countplot(data=d, x="vista", hue="diagnostico", ax=ejes[1])
ejes[1].set(title="Recuento por posición y diagnóstico", xlabel="", ylabel="pacientes")

fig.suptitle("La posición de la placa predice el diagnóstico por sí sola: riesgo de atajo",
             y=1.03, weight="bold", color="crimson")
plt.tight_layout()
plt.show()

for v in ["AP", "PA"]:
    sub = d[d.vista == v]
    print(f"{v}: n={len(sub):,}  tasa de opacidad {sub.etiqueta.mean():.1%}")

**Por qué ocurre y por qué importa.** Una placa PA se toma de pie, con el
paciente colaborando. Una AP se toma portátil, en la cama, a quien no puede
levantarse: pacientes más graves. La posición no causa la neumonía, pero
**delata la gravedad del paciente**.

El riesgo concreto es el *aprendizaje por atajo*: la red puede aprender a
reconocer las marcas de una placa portátil en vez de mirar el pulmón. Acertaría
mucho en este dataset y fallaría en un hospital con otra mezcla de posiciones.

## 5. Sexo: el contraste útil, porque aquí no hay señal

In [ ]:
fig, eje = plt.subplots(figsize=(6, 4))

tasa_s = d.groupby("sexo").etiqueta.mean().mul(100).reset_index()
sns.barplot(data=tasa_s, x="sexo", y="etiqueta", ax=eje, hue="sexo", legend=False)
eje.axhline(d.etiqueta.mean() * 100, color="black", ls=":", lw=1.5)
eje.set(title="Tasa de opacidad por sexo: sin diferencia relevante",
        xlabel="", ylabel="% con opacidad", ylim=(0, 45))
for c in eje.containers:
    eje.bar_label(c, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()

## 6. Correlaciones

In [ ]:
num = d[["edad_num", "bytes", "n_cajas", "etiqueta"]].copy()
num["es_AP"] = (d.vista == "AP").astype(int)
num["es_hombre"] = (d.sexo == "M").astype(int)
num = num.rename(columns={"edad_num": "edad", "bytes": "peso_archivo",
                          "n_cajas": "num_cajas"})

fig, eje = plt.subplots(figsize=(7, 5.5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=eje, square=True, cbar_kws={"shrink": 0.8})
eje.set_title("Correlaciones con la etiqueta", weight="bold")
plt.tight_layout()
plt.show()

print(num.corr()["etiqueta"].round(3).sort_values(ascending=False).to_string())

`num_cajas` correlaciona 0,93 porque **se deriva de la etiqueta**: no es un
predictor, es la etiqueta contada de otra forma. Se excluye de cualquier uso
como variable de entrada.

`peso_archivo` correlaciona −0,20: las placas sin hallazgos comprimen mejor
porque tienen menos textura. Es un segundo atajo potencial, más débil que la
posición pero del mismo tipo.

## 7. Cajas por paciente positivo

In [ ]:
fig, eje = plt.subplots(figsize=(6, 4))

sns.countplot(data=d[d.etiqueta == 1], x="n_cajas", ax=eje, hue="n_cajas", legend=False)
eje.set(title="Cajas de opacidad por paciente positivo",
        xlabel="número de cajas", ylabel="pacientes")
for c in eje.containers:
    eje.bar_label(c, padding=3, fontsize=9)

plt.tight_layout()
plt.show()

print(d[d.etiqueta == 1].n_cajas.value_counts().sort_index().to_dict())
print()
print("Por esto el CSV de etiquetas tiene 30 227 filas y no 26 684:")
print("un paciente positivo aporta una fila por cada caja.")

## 8. Conclusiones y consecuencias para el proyecto

| Hallazgo | Consecuencia |
| :--- | :--- |
| 22,5% de positivos, no 50% | Partición estratificada, ponderación de clases y umbral elegido sobre la curva ROC. Ya implementado. |
| **AP tiene 38,3% de opacidad frente a 9,3% en PA** | Riesgo de aprendizaje por atajo. Se documenta como sesgo y se verificará midiendo el rendimiento por separado en cada posición. |
| 4,78% de pacientes menores de 18 años | El alcance declara "adultos": hay que corregir la afirmación o filtrar el conjunto. |
| 5 pacientes con edad entre 148 y 155 años | Errores de dato: se excluyen o se marcan. |
| Sin valores faltantes; todas las imágenes 1024×1024 | No hace falta imputar ni homogeneizar tamaños. |
| El peso del archivo correlaciona −0,20 con la etiqueta | Segundo atajo potencial, del mismo tipo que la posición. |

**Qué cambia en el pipeline.** El análisis no obliga a rehacer la preparación de
datos, que ya contemplaba el desbalance y la partición por paciente. Lo que
añade son dos puntos de verificación para el informe final: medir el
rendimiento **por separado en AP y en PA**, y decidir de forma explícita qué se
hace con el 4,78% pediátrico.
